In [2]:
import geopandas as gpd
import xml.etree.ElementTree as ET
from shapely.geometry import Point
from glob import glob

In [3]:
pfad = "./afis_xml/*.xml"

In [4]:
hf = []
lf = []
po = []

ET.register_namespace("", "http://www.adv-online.de/namespaces/adv/gid/7.1")
ET.register_namespace("gmd", "http://www.opengis.net/gml/3.2")

for f in glob(pfad):
    tree = ET.parse(f)
    root = tree.getroot()
    hf += root.findall(
        './/{http://www.adv-online.de/namespaces/adv/gid/7.1}AX_Hoehenfestpunkt')
    lf += root.findall(
        './/{http://www.adv-online.de/namespaces/adv/gid/7.1}AX_Lagefestpunkt')
    po += root.findall(
        './/{http://www.adv-online.de/namespaces/adv/gid/7.1}AX_PunktortAU')

In [5]:
len(hf), len(lf), len(po)

(1671, 4498, 37593)

In [6]:
liste = {}
for i in po:
    idtag = i.find(
        './/{http://www.adv-online.de/namespaces/adv/gid/7.1}istTeilVon')
    if idtag is None:
        continue
    id = idtag.attrib['{http://www.w3.org/1999/xlink}href']
    if not id in liste:
        liste[id] = [i]
    else:
        liste[id].append(i)

In [7]:
for j in list(liste.items()):
    print(j[0])
    for i in j[1]:
        pt = i.find(
            './/{http://www.opengis.net/gml/3.2}Point')
        if ('srsName' in pt.attrib):
            print(pt.attrib['srsName'])
        else:
            # print(ET.tostring(pt, encoding='unicode'))
            print("Kein srsName")
        print(i.find(
            './/{http://www.opengis.net/gml/3.2}pos').text)

urn:adv:oid:DESHPDHK0000175y
urn:adv:crs:DE_DHHN12_NOH
67.560
urn:adv:crs:DE_DHHN92_NH
67.550
urn:adv:crs:DE_DHDN_3GK3_SH210
595144.200 5968377.630
Kein srsName
595036.530 5966428.900
urn:adv:crs:DE_Bessel_h
68.470
urn:adv:crs:ETRS89_h
107.100
urn:adv:crs:DE_DHHN2016_NH
67.539
urn:adv:oid:DESHPDHK0000175I
urn:adv:crs:DE_DHHN12_NOH
66.100
urn:adv:crs:DE_DHHN92_NH
66.090
urn:adv:crs:DE_DHDN_3GK3_SH210
595144.470 5968347.210
Kein srsName
595036.800 5966398.500
urn:adv:crs:DE_Bessel_h
67.010
urn:adv:crs:ETRS89_h
105.640
urn:adv:crs:DE_DHHN2016_NH
66.079
urn:adv:oid:DESHPDHK0000175P
urn:adv:crs:DE_DHHN12_NOH
66.290
urn:adv:crs:DE_DHHN92_NH
66.280
urn:adv:crs:DE_DHDN_3GK3_SH210
595144.350 5968360.500
Kein srsName
595036.690 5966411.780
urn:adv:crs:DE_Bessel_h
67.200
urn:adv:crs:ETRS89_h
105.830
urn:adv:crs:DE_Soldner-Rathkruegen
26548.760 2379.810
urn:adv:crs:DE_DHHN2016_NH
66.269
urn:adv:oid:DESHPDHK0000175c
urn:adv:crs:DE_DHHN12_NOH
35.710
urn:adv:crs:DE_DHHN92_NH
35.630
urn:adv:crs:DE_DHD

In [8]:
print(ET.tostring(lf[10], encoding='unicode'))

<AX_Lagefestpunkt xmlns="http://www.adv-online.de/namespaces/adv/gid/7.1" xmlns:gmd="http://www.opengis.net/gml/3.2" xmlns:ns2="http://www.w3.org/1999/xlink" gmd:id="DESHPDHK0000176Q">
					<gmd:identifier codeSpace="http://www.adv-online.de/">urn:adv:oid:DESHPDHK0000176Q</gmd:identifier>
					<lebenszeitintervall>
						<AA_Lebenszeitintervall>
							<beginnt>2011-03-30T14:05:11Z</beginnt>
						</AA_Lebenszeitintervall>
					</lebenszeitintervall>
					<modellart>
						<AA_Modellart>
							<advStandardModell>DFGM</advStandardModell>
						</AA_Modellart>
					</modellart>
					<punktkennung>212801320</punktkennung>
					<gemeinde>
						<AX_Gemeindekennzeichen>
							<land>01</land>
							<regierungsbezirk>0</regierungsbezirk>
							<kreis>62</kreis>
							<gemeinde>004</gemeinde>
						</AX_Gemeindekennzeichen>
					</gemeinde>
					<katasteramt>
						<AX_Dienststelle_Schluessel>
							<land>01</land>
							<stelle>0001</stelle>
						</AX_Dienststelle_Schluessel>
					</k

In [11]:
lagefestpunkte = []
for l in lf:
    id = l.find('{http://www.opengis.net/gml/3.2}identifier').text
    print(id)
    xy = None
    h = None
    for i in liste[id]:
        pt = i.find(
            './/{http://www.opengis.net/gml/3.2}Point')
        k = pt.find(
            './/{http://www.opengis.net/gml/3.2}pos').text
        if ('srsName' in pt.attrib):
            print(pt.attrib['srsName'], '\t-\t', k)
            if pt.attrib['srsName'] == 'urn:adv:crs:DE_DHHN2016_NH':
                h = k
            if pt.attrib['srsName'] == 'urn:adv:crs:DE_DHDN_3GK3_SH210':
                xy = k.split(' ')
        else:
            # print(ET.tostring(pt, encoding='unicode'))
            xy = k.split(' ')
            print("Kein srsName", '\t-\t', k)
    vermarkung = l.find(
        '{http://www.adv-online.de/namespaces/adv/gid/7.1}punktvermarkung').text
    relHoehe = l.find(
        '{http://www.adv-online.de/namespaces/adv/gid/7.1}relativeHoehe')
    name = l.find(
        '{http://www.adv-online.de/namespaces/adv/gid/7.1}nameLagebeschreibung')
    if relHoehe is not None:
        relHoehe = relHoehe.text
    if name is not None:
        name = name.text
    if (xy is not None) and (h is not None):
        lagefestpunkte.append([id, name, vermarkung, relHoehe, float(xy[0]), float(
            xy[1]), float(h), Point(float(xy[0]), float(xy[1]), float(h))])

urn:adv:oid:DESHPDHK0000175I
urn:adv:crs:DE_DHHN12_NOH 	-	 66.100
urn:adv:crs:DE_DHHN92_NH 	-	 66.090
urn:adv:crs:DE_DHDN_3GK3_SH210 	-	 595144.470 5968347.210
Kein srsName 	-	 595036.800 5966398.500
urn:adv:crs:DE_Bessel_h 	-	 67.010
urn:adv:crs:ETRS89_h 	-	 105.640
urn:adv:crs:DE_DHHN2016_NH 	-	 66.079
urn:adv:oid:DESHPDHK0000175P
urn:adv:crs:DE_DHHN12_NOH 	-	 66.290
urn:adv:crs:DE_DHHN92_NH 	-	 66.280
urn:adv:crs:DE_DHDN_3GK3_SH210 	-	 595144.350 5968360.500
Kein srsName 	-	 595036.690 5966411.780
urn:adv:crs:DE_Bessel_h 	-	 67.200
urn:adv:crs:ETRS89_h 	-	 105.830
urn:adv:crs:DE_Soldner-Rathkruegen 	-	 26548.760 2379.810
urn:adv:crs:DE_DHHN2016_NH 	-	 66.269
urn:adv:oid:DESHPDHK0000175X
urn:adv:crs:DE_DHHN92_NH 	-	 64.750
urn:adv:crs:DE_DHDN_3GK3_SH210 	-	 594828.500 5968342.150
Kein srsName 	-	 594720.960 5966393.440
urn:adv:crs:DE_Bessel_h 	-	 65.670
urn:adv:crs:ETRS89_h 	-	 104.301
urn:adv:crs:DE_DHHN2016_NH 	-	 64.739
urn:adv:oid:DESHPDHK0000175c
urn:adv:crs:DE_DHHN12_NOH 	-	 35

In [12]:
df_lf = gpd.GeoDataFrame(lagefestpunkte, columns=[
                         'id', 'name', 'vermarkung', 'relHoehe', 'x', 'y', 'z',  'geometry'])
df_lf.crs = 'EPSG:25832'
df_lf.to_file('sh.gpkg', driver='GPKG')